# Appels LLM et création de prompts enrichis

Bienvenue dans ce TP consacré aux **appels aux modèles de langage (LLM)** et à la **construction de prompts enrichis**.

Dans ce TP, vous allez pratiquer concrètement comment :
- poser une question simple à un modèle de langage,
- dialoguer avec un modèle (conversation),
- enrichir un prompt avec du **contexte supplémentaire** afin d’obtenir des réponses plus précises.

👉 **Objectif pédagogique** :  
Comprendre *comment* on interagit avec un LLM, avant d’aller plus loin vers les systèmes RAG (Retrieval-Augmented Generation).

---

## Ce que vous allez apprendre dans ce TP

À la fin de ce TP, vous serez capables de :

- Appeler un LLM avec une **question simple**.
- Appeler un LLM avec une **liste de messages** (conversation).
- Comprendre pourquoi **ajouter du contexte** dans un prompt améliore la qualité des réponses.
- Exécuter ces appels **en local, sans Internet**, grâce à **Ollama**.

---

## Table des matières
- [1 - Comprendre les fonctions d’appel aux LLM](#1)
  - [1.1 `generate_with_single_input`](#1-1)
  - [1.2 `generate_with_multiple_input`](#1-2)
- [2 - Intégrer des données dans un prompt](#2)
  - [2.1 Comprendre la structure des données](#2-1)
  - [2.2 Construire un prompt enrichi](#2-2)

In [ ]:
from utils import (
    generate_with_single_input, 
    generate_with_multiple_input
)

## 1 - Comprendre les fonctions d’appel aux LLM

Dans ce TP, nous utiliserons **deux fonctions principales** pour interagir avec un modèle de langage (LLM) :

- **`generate_with_single_input`** : pour poser **une seule question** au modèle.
- **`generate_with_multiple_input`** : pour gérer une **conversation**, avec plusieurs messages successifs.

Ces deux fonctions permettent de couvrir les usages les plus courants des LLM :
- question/réponse simple,
- discussion structurée (assistant conversationnel).

---

👉 **Dans ce TP, ce n’est pas le cas** :

- Les appels sont faits **localement** sur votre machine.
- Le modèle de langage est exécuté via **Ollama**.
- **Aucune clé API n’est nécessaire**.
- Le TP fonctionne **hors connexion Internet**.


- Ici tout se passe **en local**, ce qui est plus simple, plus fiable et gratuit.

---

<a id='1-1'></a>
## 1.1 `generate_with_single_input`

La fonction `generate_with_single_input` permet de générer du texte à partir **d’un seul prompt**.

C’est la forme **la plus simple** d’appel à un LLM :

> **Je pose une question → le modèle me répond**

Nous allons commencer avec **peu de paramètres**, afin de bien comprendre le mécanisme de base.
Les paramètres plus avancés (température, top-p, etc.) seront abordés plus tard dans le cours.

---

### Paramètres principaux

- **`prompt`** *(str)*  
  La question ou l’instruction envoyée au modèle.

- **`max_tokens`** *(int)*  
  Le nombre maximum de tokens générés dans la réponse.

- **`model`** *(str, optionnel)*  
  Le nom du modèle Ollama utilisé  
  (par défaut : le modèle local configuré sur la machine).

In [21]:
# Example call
output = generate_with_single_input(
    prompt="Quelle est la capitale du Maroc ?"
)

print("Role:", output['role'])
print("Content:", output['content'])

Role: assistant
Content: La capitale du Maroc est Rabat.


<a id='1-2'></a>
### 1.2 `generate_with_multiple_input`

Cette fonction permet de dialoguer avec un LLM en plusieurs étapes  
(comme une conversation).

Chaque message est défini par :
- `role` : le rôle du message (`system`, `user`, `assistant`)
- `content` : le texte du message

On l’utilise quand on veut :
- donner un **contexte global** (rôle system)
- poser plusieurs questions successives
- guider le comportement du modèle


In [22]:
# Example call
messages = [
    {'role': 'user', 'content': 'Bonjour, qui a remporté la coupe d\'afrique des nations en 2019 ?'},
    {'role': 'assistant', 'content': 'L\'Algerie a remporté la Coupe d\'Afrique des Nations 2019.'},
    {'role': 'user', 'content': 'Qui était le capitaine de l\'équipe ?'}
]

output = generate_with_multiple_input(
    messages=messages,
    max_tokens=100
)

print("Role:", output['role'])
print("Content:", output['content'])

Role: assistant
Content: Le capitaine de l'équipe d'Algérie vainqueur de la Coupe d'Afrique des Nations 2019 était Riyad Mahrez, un joueur de football algérien qui évolue au poste d'ailier droit au club anglais Manchester City.


In [23]:
from utils import OllamaConfig, healthcheck, list_models

cfg = OllamaConfig()

if not healthcheck(cfg):
    raise RuntimeError("❌ Ollama n’est pas lancé. Lance `ollama serve` puis réessaie.")

print("✅ Ollama prêt sur :", cfg.host)
print("Modèle LLM utilisé :", cfg.model)
print("Modèles disponibles (extrait) :", list_models(cfg)[:10])


✅ Ollama prêt sur : http://localhost:11434
Modèle LLM utilisé : llama3:latest
Modèles disponibles (extrait) : ['gemma3:12b', 'nomic-embed-text:latest', 'deepseek-r1:1.5b', 'deepseek-r1:latest', 'llama3:latest', 'llama2:latest', 'mxbai-embed-large:latest', 'gemma2:2b']


Pour l'utiliser, Même exemple que précédemment, avec un historique de conversation.

In [28]:
messages = [
    {"role": "user", "content": "Bonjour, qui a remporté la Coupe du Monde FIFA en 1998 ?"},
    {"role": "assistant", "content": "La France a remporté la Coupe du Monde FIFA 1998."},
    {"role": "user", "content": "Qui était le capitaine ?"}
]


In [29]:
from utils import generate_with_multiple_input

response = generate_with_multiple_input(
    messages,
    model="llama3:latest",   # ou cfg.model
    max_tokens=300
)

In [30]:
print(response)

{'role': 'assistant', 'content': "Le capitaine de l'équipe de France vainqueur de la Coupe du Monde FIFA 1998 était Didier Deschamps."}


- Remarque : la réponse retournée contient plusieurs champs. Pour accéder au texte généré par le modèle, on utilise simplement :

`response["content"]`

In [31]:
print(response["content"])


Le capitaine de l'équipe de France vainqueur de la Coupe du Monde FIFA 1998 était Didier Deschamps.


<a id='2'></a>
## 2 - Intégrer des données dans un prompt LLM

Dans cette section, nous allons apprendre à intégrer des données externes dans un prompt
avant de l’envoyer à un modèle de langage (LLM).

Nous utiliserons un petit jeu de données au format JSON contenant des informations
sur des maisons.  
L’objectif est de comprendre le principe d’**augmentation de prompt**, base des systèmes RAG.

<a id='2-1'></a>
### 2.1 Comprendre la structure des données

Jetons un coup d’œil à la structure des données.  
Il s’agit d’un petit dataset : une liste contenant un dictionnaire par maison.

In [32]:
house_data = [
    {
        "address": "123 Rue des Baobabs",
        "city": "Dakar",
        "state": "DK",
        "zip": "11000",
        "bedrooms": 3,
        "bathrooms": 2,
        "square_feet": 1500,
        "price": 230000,
        "year_built": 1998
    },
    {
        "address": "456 Avenue de l’Indépendance",
        "city": "Thiès",
        "state": "TH",
        "zip": "21000",
        "bedrooms": 4,
        "bathrooms": 3,
        "square_feet": 2500,
        "price": 320000,
        "year_built": 2005
    }
]

<a id='2-2'></a>
### 2.2 Création du prompt

Commençons par construire le prompt.  
La première étape consiste à définir la manière dont les données seront présentées
au modèle.

In [33]:
# Création d’un format texte pour décrire les maisons

def house_info_layout(houses):
    # Chaîne vide pour stocker le résultat
    layout = ""
    # Parcours de chaque maison
    for house in houses:
        # Ajout des informations de la maison sous forme de texte
        layout += (
            f"Maison située à {house['address']}, {house['city']}, {house['state']} {house['zip']} avec "
            f"{house['bedrooms']} chambres, {house['bathrooms']} salles de bain, "
            f"une surface de {house['square_feet']} pieds carrés, au prix de ${house['price']}, "
            f"construite en {house['year_built']}.\n"
        )  # Retour à la ligne pour séparer les maisons
    return layout

In [34]:
# Vérification du format des données
print(house_info_layout(house_data))

Maison située à 123 Rue des Baobabs, Dakar, DK 11000 avec 3 chambres, 2 salles de bain, une surface de 1500 pieds carrés, au prix de $230000, construite en 1998.
Maison située à 456 Avenue de l’Indépendance, Thiès, TH 21000 avec 4 chambres, 3 salles de bain, une surface de 2500 pieds carrés, au prix de $320000, construite en 2005.



Maintenant, créez une fonction qui génère le **prompt** à envoyer au modèle (LLM).

Cette fonction prendra :
- la question de l’utilisateur (`query`)
- les données disponibles sur les maisons (`houses`)

Objectif : construire un prompt clair qui aide le modèle à répondre correctement à la question.

In [35]:
def generate_prompt(query, houses):
    # Le code est modulaire : on peut fournir tout ou partie des données
    houses_layout = house_info_layout(houses)

    # Prompt construit avec les données + la question utilisateur
    PROMPT = f"""
Utilise les informations suivantes sur les maisons pour répondre à la question.
{houses_layout}
Question : {query}
    """
    return PROMPT

In [36]:
print(generate_prompt("Quelle est la maison la plus chère ?", houses=house_data))



Utilise les informations suivantes sur les maisons pour répondre à la question.
Maison située à 123 Rue des Baobabs, Dakar, DK 11000 avec 3 chambres, 2 salles de bain, une surface de 1500 pieds carrés, au prix de $230000, construite en 1998.
Maison située à 456 Avenue de l’Indépendance, Thiès, TH 21000 avec 4 chambres, 3 salles de bain, une surface de 2500 pieds carrés, au prix de $320000, construite en 2005.

Question : Quelle est la maison la plus chère ?
    


- Appel du LLM avec et sans données augmentées

In [37]:
query = "Quelle est la maison la plus chère ? Et la plus grande ?"

# Sans données (question simple)
query_without_house_info = generate_with_single_input(
    prompt=query,
    role="user"
)

# Avec données (prompt augmenté)
enhanced_query = generate_prompt(query, houses=house_data)
query_with_house_info = generate_with_single_input(
    prompt=enhanced_query,
    role="assistant"
)


In [38]:
# Sans données sur les maisons
print(query_without_house_info["content"])


Excellentes questions !

La maison la plus chère au monde, selon les estimations, est le "Antilia" de Mukesh Ambani, un magnat indien du pétrole. Construite à Mumbai (Inde), cette demeure privée a coûté environ 1 milliard de dollars (soit environ 900 millions d'euros) ! Elle mesure 27 étages et compte plus de 400 pièces, dont des salles de cinéma, des piscines, des courts de tennis, etc. La famille Ambani y vit avec ses 600 employés.

Quant à la maison la plus grande au monde, c'est le "Biltmore Estate" situé à Asheville (Caroline du Nord, États-Unis). Construit en 1895 pour George Vanderbilt, ce château de style Renaissance française mesure environ 30 000 mètres carrés et compte 250 pièces ! Il est considéré comme l'un des plus grands maisons privées au monde. Le Biltmore Estate est également célèbre pour son parc de 8 hectares, ses jardins à thème et sa collection d'art et d'objets précieux.

Il convient de noter que ces chiffres peuvent varier en fonction des sources et des définiti

In [39]:
# Avec données sur les maisons  
print(query_with_house_info['content'])

En répondant à la question, je peux dire que :

* La maison la plus chère est celle située à 456 Avenue de l’Indépendance, Thiès, TH 21000 avec un prix de $320000.
* La maison la plus grande est également celle située à 456 Avenue de l’Indépendance, Thiès, TH 21000 avec une surface de 2500 pieds carrés.


Bravo 🎉  
Vous avez terminé ce TP d’introduction sur l’appel aux modèles de langage (LLM)
et l’augmentation de prompts avec des données externes.

Continuez comme ça 💪
